## Imports

In [1]:
from qsopt import * 
import numpy as np
import jax.numpy as jnp
import optax
import matplotlib.pyplot as plt

## Define experimental parameters

In [2]:
gm = 0.03 * 2 * np.pi
inverse_pulse_width = 0.1 * gm

# Define custom physical constants
custom_constants = PhysicalConstants(
    chi = 0.5 * gm,                    # Dispersive coupling
    photon_cavity_coupling = gm,  # Photon-cavity coupling
    inverse_pulse_width = inverse_pulse_width      # Inverse pulse width
)

# Define custom system dimensions
custom_dims = SystemDimensions(
    cavity_levels=2,
    qubit_levels=2,
    field_levels=2
)

# Define measurement protocol
custom_measurement = MeasurementProtocol(
    measurement_times=None,  # Use interval mode
    initial_time=-7.0/inverse_pulse_width,
    final_time=7.0/inverse_pulse_width,
    time_interval=5.0/inverse_pulse_width,
    initial_time_uncertainty=2.0/inverse_pulse_width
)

# Define initial state configuration (SINGLE_PHOTON)
initial_state = InitialStateConfig(
    state_type=InitialStateType.SINGLE_PHOTON
)

# Define noise configuration
noise_config = NoiseConfiguration(
    depolarizing = 0.0001,  
    dephasing = 0.0001,      
    relaxation = 0.0001
)

# Create parameters with custom configuration
exp_parameters = ExperimentalParameters(
    physical_constants=custom_constants,
    system_dims=custom_dims,
    measurement=custom_measurement,
    initial_state=initial_state,
    noise_config=noise_config,
    random_seed=42
)

print(exp_parameters)

SYSTEM DIMENSIONS
  Cavity levels:             2
  Qubit levels:              2
  Field levels:              2
  Total dimension:           8
PHYSICAL CONSTANTS
  Chi:                    0.0942
  Photon cavity coupling: 0.1885
  Inverse pulse width:    0.0188
MEASUREMENT PROTOCOL
  Mode:                 Interval-based
  Initial time:         -371.3615
  Final time:           371.3615
  Time interval:        265.2582
  Number of measurements:      4
  Computed times:       [-371.36153388108914, -106.1032953945969, 159.15494309189535, 424.4131815783876]
  Initial time uncertainty: 106.1033
INITIAL STATE
  Type:                 single_photon
NOISE MODEL
  Depolarizing rate:      0.0001
  Dephasing rate:         0.0001
  Relaxation rate:        0.0001
  Custom operators:     None
SYSTEM STATUS
  Configuration:        VALID


## Define trainable parameters

In [3]:
parameters = TrainableParameters()
# Add rotation angles (always needed)
parameters.add_rotation_angles(
    names=['theta1', 'theta2'],
    initial_values=[np.pi/2, -np.pi/2],
    trainable=[False, False]
)

# Add trainable time_interval
# NOTE: This will OVERRIDE the value from experimental_parameters
parameters.add_measurement_interval(
    names='time_interval',
    initial_values=1.0/inverse_pulse_width,      # Starting value for optimization
    min_interval=0.000001,        # Minimum allowed value (must be > 0)
    trainable=True           # Enable optimization
)

print(parameters)

Trainable Parameters: 3
  Rotation Angles:
    theta1: 1.5708 rad (90.00°) [FIXED]
    theta2: -1.5708 rad (-90.00°) [FIXED]
  Measurement Times:
    time_interval: 53.0516


## Define experiment

In [4]:
experiment = SingleQubitExperiment(exp_parameters, parameters)

## Test Single Simulation

In [5]:
results = experiment.run_simulation(batch_size=10)
print(results)

MODE: Single Simulation
  Current Parameters:
     theta1: 1.570796 rad (90.00°)
     theta2: -1.570796 rad (-90.00°)
  Detection Probabilities:
     P(with photon):    0.953165
     P(without photon): 0.578462
     Contrast:          0.374703


## Run Optimization

In [ ]:
history = experiment.optimize(
    num_steps=70,
    verbose=True,
    tolerance=1e-7,
    batch_size=10,
)

Configuration:
    Max iterations: 70
    Batch size: 10
    Convergence tolerance: 1.00e-07
    Initial rotation parameters: theta1=1.571 rad [FIXED], theta2=-1.571 rad [FIXED]
    Initial time interval: 53.051648
    Optimizer: GradientTransformationExtraArgs
    Measurement uncertainty: ±106.103
Step  theta1      theta2      Δt          Contrast    Grad Norm
----------------------------------------------------------------------
0     1.570796    -1.570796   53.051648   0.000015    5.19e-06    
10    1.570796    -1.570796   53.055983   -0.024291   2.66e-03    
20    1.570796    -1.570796   53.040443   -0.022835   3.07e-03    
30    1.570796    -1.570796   53.053198   -0.024522   6.02e-03    


Exception ignored in: <function _xla_gc_callback at 0x0000014C780FADE0>
Traceback (most recent call last):
  File "c:\Users\simon\miniconda3\envs\quantum\Lib\site-packages\jax\_src\lib\__init__.py", line 96, in _xla_gc_callback
    def _xla_gc_callback(*args):
KeyboardInterrupt: 


In [7]:
print(history)

MODE: Optimization
     Total iterations: 1
     Best epoch:        1
     Converged: True
     Final gradient norm: 0.000000e+00
  Best Parameters:
     theta1: 1.570796 rad (90.00°)
     theta2: -1.570796 rad (-90.00°)
  Detection Probabilities:
     P(with photon):    0.973022
     P(without photon): 0.000000
     Contrast:          0.973022
